# Scenario 12: Enterprise OTel Pattern — Fully Manual Spans

This notebook mirrors the **exact pattern from the customer's sample script**: every span — workflow, retriever, LLM, and tool — is created by hand with `tracer.start_as_current_span(...)`. There are no OpenInference instrumentors, no Galileo span helpers, and no auto-instrumentation of the OpenAI client.

The only automatic component is `RequestsInstrumentor`, which captures outbound HTTP calls as child spans (matching the customer's `init_telemetry()` setup).

## How this compares to Scenarios 10 and 11

| Aspect | Scenario 10 | Scenario 11 | Scenario 12 (this notebook) |
|---|---|---|---|
| Span processor | `GalileoSpanProcessor` | Raw `BatchSpanProcessor` + `OTLPSpanExporter` | Raw `BatchSpanProcessor` + `OTLPSpanExporter` |
| LLM spans | Auto (OpenInference `OpenAIInstrumentor`) | Auto (OpenInference `OpenAIInstrumentor`) | **Manual** (`tracer.start_as_current_span`) |
| Custom spans | Galileo schema classes | `gen_ai.*` attributes | `gen_ai.*` attributes |
| HTTP instrumentation | None | None | `RequestsInstrumentor` |
| OpenInference dependency | `openinference-instrumentation-openai` | `openinference-instrumentation-openai` | **None** |
| Coupling to any library | Galileo SDK | OpenInference | **Zero** — pure OTel only |

By the end, you'll understand how to:

1. Build a **standard OTel pipeline** aimed at Galileo with no vendor dependencies in the span path
2. Create **manual LLM spans** with the `gen_ai.*` attributes Galileo needs
3. Create **workflow, retriever, and tool spans** matching the customer's naming and attribute conventions
4. Use `RequestsInstrumentor` so outbound HTTP calls appear as child spans
5. Wrap the boilerplate in **reusable decorators**
6. Flush and shut down cleanly

> **Production note:** In a FastAPI service, you'd add `FastAPIInstrumentor.instrument_app(app)` to get inbound request spans automatically. In a notebook there's no server, so we skip it.

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`


## Step 0: Load Environment Variables

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Imports and Constants

There is **no OpenInference import** here. Every dependency is either standard-library, vanilla OpenTelemetry, or the OpenAI client. The only Galileo imports are for project setup and teardown — nothing touches the span path.

### How Galileo extracts fields from OTel spans

Galileo's OTLP backend automatically maps span attributes with a fallback chain (attributes first, then span events). No per-project configuration is needed.

| Field | Attribute (primary) | Fallback |
|---|---|---|
| Input/output messages | `gen_ai.input.messages` / `gen_ai.output.messages` | Span events (`gen_ai.user.message`, `gen_ai.assistant.message`) |
| System instructions | `gen_ai.system_instructions` | First system message in `gen_ai.input.messages` |
| Token metrics | `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` | OpenInference `llm.token_count.prompt` / `.completion` |
| Context/documents | Retriever span output (list of documents) | Extracted automatically from retriever spans |
| Ground truth | `galileo.dataset.input` / `galileo.dataset.output` | Resource attributes for evaluation metrics |

### Galileo span classification attributes

| Span type | Primary discriminator | Input / Output |
|---|---|---|
| LLM | `gen_ai.operation.name` = `"chat"` | `gen_ai.request.model`, `gen_ai.input.messages`, `gen_ai.output.messages`, `gen_ai.usage.input_tokens` / `.output_tokens` |
| Retriever | `db.operation` = `"search"` | `gen_ai.input.messages`, `gen_ai.output.messages` with `{"documents": [...]}` |
| Tool | `gen_ai.operation.name` = `"execute_tool"` | `gen_ai.tool.name`, `gen_ai.tool.call.arguments` / `.result` |
| Workflow | (parent span) | `gen_ai.input.messages`, `gen_ai.output.messages` |


In [2]:
import functools
import json
import os
from urllib.parse import urljoin

import openai
from galileo import galileo_context
from galileo.config import GalileoPythonConfig
from galileo.projects import delete_project
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

PROJECT_NAME = 'GalileoEval_S12_ManualOTel'
LOG_STREAM_NAME = 'manual-otel'
SERVICE_NAME = 'galileo-manual-otel-demo'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, SERVICE_NAME, MODEL

('GalileoEval_S12_ManualOTel',
 'manual-otel',
 'galileo-manual-otel-demo',
 'gpt-4o-mini')

## Step 1: Initialize Galileo (project + log stream only)

`galileo_context.init()` creates the project and log stream. We read back the API URL and API key to wire into the raw OTLP exporter. After this cell, the Galileo SDK is not involved in the tracing path.

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

GALILEO_API_URL = str(config.api_url)
GALILEO_API_KEY = config.api_key.get_secret_value() if config.api_key else None

OTEL_TRACES_ENDPOINT = os.environ.get(
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT',
    urljoin(GALILEO_API_URL if GALILEO_API_URL.endswith('/') else GALILEO_API_URL + '/', 'otel/traces'),
)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    log_stream_url = 'Galileo context initialized, but no log stream URL was returned.'
    print(log_stream_url)

print(f'\nOTLP endpoint -> {OTEL_TRACES_ENDPOINT}')

https://console.demo-v2.galileocloud.io/project/9ec42db0-d40c-4ee0-9f28-e6f29a83288f
https://console.demo-v2.galileocloud.io/project/9ec42db0-d40c-4ee0-9f28-e6f29a83288f/log-streams/470b0435-70fd-4aca-953d-7b208f5dc7bb

OTLP endpoint -> https://api.demo-v2.galileocloud.io/otel/traces


## Step 2: Build the OTel Pipeline

This matches the customer's `init_telemetry(app)` function:

1. `TracerProvider` with a service `Resource`
2. `OTLPSpanExporter` pointing at the Galileo OTLP endpoint with auth headers
3. `BatchSpanProcessor` wrapping the exporter
4. `RequestsInstrumentor` so outbound `requests.*` calls become child spans automatically

The only difference from the customer's code is that we skip `FastAPIInstrumentor` (no server in a notebook) and we get a `tracer` instance to create manual spans with.

In [4]:
def configure_tracing():
    resource = Resource.create({
        'service.name': SERVICE_NAME,
        'service.version': '1.0.0',
        'deployment.environment': 'notebook',
    })

    tracer_provider = TracerProvider(resource=resource)

    exporter = OTLPSpanExporter(
        endpoint=OTEL_TRACES_ENDPOINT,
        headers={
            'Galileo-API-Key': GALILEO_API_KEY,
            'project': PROJECT_NAME,
            'logstream': LOG_STREAM_NAME,
        },
    )

    span_processor = BatchSpanProcessor(exporter)
    tracer_provider.add_span_processor(span_processor)
    trace.set_tracer_provider(tracer_provider)

    # HTTP auto-instrumentation (matches customer's RequestsInstrumentor().instrument())
    req_instrumentor = RequestsInstrumentor()
    try:
        req_instrumentor.uninstrument()
    except Exception:
        pass
    req_instrumentor.instrument(tracer_provider=tracer_provider)

    return tracer_provider, span_processor, req_instrumentor


def force_flush(timeout_millis: int = 40000) -> None:
    span_processor.force_flush(timeout_millis)


tracer_provider, span_processor, req_instrumentor = configure_tracing()
tracer = trace.get_tracer('fin-workflow')  # matches customer's tracer name
client = openai.OpenAI()

print('Pure OTel pipeline ready (no OpenInference):')
print(f'  Service      -> {SERVICE_NAME}')
print(f'  Endpoint     -> {OTEL_TRACES_ENDPOINT}')
print(f'  Tracer       -> fin-workflow')
print('  Instrumentor -> RequestsInstrumentor (HTTP only, no LLM auto-instrumentation)')

Attempting to uninstrument while already uninstrumented


Pure OTel pipeline ready (no OpenInference):
  Service      -> galileo-manual-otel-demo
  Endpoint     -> https://api.demo-v2.galileocloud.io/otel/traces
  Tracer       -> fin-workflow
  Instrumentor -> RequestsInstrumentor (HTTP only, no LLM auto-instrumentation)


## Step 3: Manual LLM Span

Without `OpenAIInstrumentor`, we create the LLM span ourselves. Galileo extracts fields from span attributes automatically:

- `gen_ai.operation.name` = `"chat"` — the span type discriminator
- `gen_ai.request.model` — model name shown in the console
- `gen_ai.input.messages` / `gen_ai.output.messages` — conversation content
- `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` — token metrics (note: OTel semconv uses `input_tokens`/`output_tokens`, not `prompt_tokens`/`completion_tokens`)

This is the pattern the customer would use for any LLM call — whether via the OpenAI SDK, a raw HTTP POST, or an internal model gateway.

In [5]:
user_messages = [
    {'role': 'system', 'content': 'You are a helpful assistant that explains technical concepts clearly.'},
    {'role': 'user', 'content': 'What is OpenTelemetry and why is it useful for AI applications?'},
]

with tracer.start_as_current_span(
    'chat',
    attributes={
        'gen_ai.system': 'openai',
        'gen_ai.operation.name': 'chat',
        'gen_ai.request.model': MODEL,
        'gen_ai.input.messages': json.dumps(user_messages),
    },
) as llm_span:
    response = client.chat.completions.create(
        model=MODEL,
        messages=user_messages,
        max_tokens=200,
    )
    answer = response.choices[0].message.content

    llm_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': answer}]),
    )
    llm_span.set_attribute('gen_ai.response.model', response.model)
    llm_span.set_attribute('gen_ai.usage.input_tokens', response.usage.prompt_tokens)
    llm_span.set_attribute('gen_ai.usage.output_tokens', response.usage.completion_tokens)

force_flush()

print('Response:')
print(answer)
print(f'\nTokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out')
print(f'\n-> Manual LLM span flushed to Galileo. Check: {log_stream_url}')

Response:
OpenTelemetry is an open-source observability framework that provides a standardized way to collect, process, and export telemetry data from applications. It encompasses three main types of telemetry data: traces, metrics, and logs, allowing developers to monitor and understand the performance and behavior of their applications.

Here's why OpenTelemetry is particularly useful for AI applications:

1. **Data Collection**: AI applications typically rely on large volumes of data, both for training models and for operational monitoring. OpenTelemetry facilitates the collection of this data in a standardized way, making it easier to track how data flows through your AI systems.

2. **Performance Monitoring**: AI models can be resource-intensive. OpenTelemetry allows you to gather metrics about resource usage, such as CPU, memory, and GPU utilization, which can help identify bottlenecks or inefficiencies in your AI applications. This is critical for optimizing performance.

3. **T

## Step 4: Customer-Style Nested Trace — workflow → retriever → LLM → tool

This cell mirrors the customer's sample script as closely as possible. Every span is created with `tracer.start_as_current_span(...)` — including the LLM call. The span names follow the customer's `workflow:{name}`, `tool:{step_name}`, `retriever:{endpoint}` convention.

The business-specific attributes (`workflow.name`, `usecase.id`, `sys.id`, `step.name`, `step.operation`, `http.url`, `http.method`) are carried as free-form OTel attributes.

In [6]:
user_question = 'What are the best practices for OpenTelemetry in LLM apps?'
workflow_name = 'research-agent'

with tracer.start_as_current_span(
    f'workflow:{workflow_name}',
    attributes={
        'gen_ai.system': 'custom-otel',
        'workflow.name': workflow_name,
        'usecase.id': 'demo-usecase-001',
        'sys.id': 'notebook-sys-id',
        'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
    },
) as workflow_span:

    # 1) Retriever span — db.operation = search is the Galileo discriminator.
    retrieved_docs = [
        {'content': 'Always use a BatchSpanProcessor in production.', 'metadata': {'source': 'otel-docs', 'score': 0.91}},
        {'content': 'Set OTEL_SERVICE_NAME so spans carry a service identity.', 'metadata': {'source': 'otel-docs', 'score': 0.87}},
        {'content': 'Use vendor headers for OTLP routing instead of per-span attributes.', 'metadata': {'source': 'galileo-docs', 'score': 0.84}},
    ]
    with tracer.start_as_current_span(
        'retriever:vector-search',
        attributes={
            'gen_ai.system': 'custom-otel',
            'db.operation': 'search',
            'http.url': 'https://vector-store.internal/query',
            'http.method': 'POST',
            'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
        },
    ) as retriever_span:
        retriever_span.set_attribute(
            'gen_ai.output.messages',
            json.dumps([{'role': 'assistant', 'content': {'documents': retrieved_docs}}]),
        )
        print(f'Retrieved {len(retrieved_docs)} documents')

    # 2) Manual LLM span — same gen_ai.* attributes as Step 3.
    context_text = '\n'.join(f'- {d["content"]}' for d in retrieved_docs)
    llm_messages = [
        {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
        {'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {user_question}'},
    ]
    with tracer.start_as_current_span(
        'chat',
        attributes={
            'gen_ai.system': 'openai',
            'gen_ai.operation.name': 'chat',
            'gen_ai.request.model': MODEL,
            'gen_ai.input.messages': json.dumps(llm_messages),
        },
    ) as llm_span:
        synthesis = client.chat.completions.create(
            model=MODEL,
            messages=llm_messages,
            max_tokens=150,
        )
        answer = synthesis.choices[0].message.content
        llm_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'assistant', 'content': answer}]))
        llm_span.set_attribute('gen_ai.response.model', synthesis.model)
        llm_span.set_attribute('gen_ai.usage.input_tokens', synthesis.usage.prompt_tokens)
        llm_span.set_attribute('gen_ai.usage.output_tokens', synthesis.usage.completion_tokens)

    # 3) Tool span — gen_ai.operation.name = execute_tool is the discriminator.
    step_name = 'format-final-answer'
    with tracer.start_as_current_span(
        f'tool:{step_name}',
        attributes={
            'gen_ai.system': 'custom-otel',
            'gen_ai.operation.name': 'execute_tool',
            'gen_ai.tool.name': step_name,
            'step.name': step_name,
            'step.operation': 'strip-whitespace',
            'gen_ai.tool.call.arguments': answer,
            'gen_ai.input.messages': json.dumps([{'role': 'tool', 'content': answer}]),
        },
    ) as tool_span:
        formatted = answer.strip()
        tool_span.set_attribute('gen_ai.tool.call.result', formatted)
        tool_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'tool', 'content': formatted}]))

    workflow_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': formatted}]),
    )

force_flush()

print(f'\nFinal answer: {formatted}')
print('\n-> One trace with workflow -> retriever -> llm -> tool spans (all manual)')
print(f'   Check: {log_stream_url}')

Retrieved 3 documents



Final answer: The best practices for OpenTelemetry in LLM apps include:

1. Always use a BatchSpanProcessor in production for efficient span processing.
2. Set the OTEL_SERVICE_NAME environment variable to ensure spans carry the correct service identity.
3. Utilize vendor headers for OTLP routing instead of relying on per-span attributes for better performance and clarity.

-> One trace with workflow -> retriever -> llm -> tool spans (all manual)
   Check: https://console.demo-v2.galileocloud.io/project/9ec42db0-d40c-4ee0-9f28-e6f29a83288f/log-streams/470b0435-70fd-4aca-953d-7b208f5dc7bb


## Step 5: Reusable Decorators

These decorators wrap the manual span pattern so application code stays clean. Each one sets the **correct Galileo discriminator attributes** for its span type.

The `@otel_llm` decorator is new compared to Scenarios 10–11: it wraps any function that calls an LLM, extracts the response, and populates the `gen_ai.*` attributes that Galileo expects.

In [7]:
def _snapshot(args, kwargs):
    return json.dumps({'args': [str(a) for a in args], 'kwargs': {k: str(v) for k, v in kwargs.items()}})


def _msg(role, content):
    return json.dumps([{'role': role, 'content': content}])


def otel_workflow(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            input_str = _snapshot(args, kwargs)
            with tracer.start_as_current_span(
                f'workflow:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'workflow.name': name,
                    'gen_ai.input.messages': _msg('user', input_str),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                span.set_attribute('gen_ai.output.messages', _msg('assistant', str(result)))
                return result
        return wrapper
    return decorator


def otel_retriever(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            query = args[0] if args else kwargs.get('query', '')
            with tracer.start_as_current_span(
                f'retriever:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'db.operation': 'search',
                    'gen_ai.input.messages': _msg('user', str(query)),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                docs = result if isinstance(result, list) else [result]
                span.set_attribute(
                    'gen_ai.output.messages',
                    json.dumps([{'role': 'assistant', 'content': {'documents': docs}}]),
                )
                return result
        return wrapper
    return decorator


def otel_tool(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            input_str = _snapshot(args, kwargs)
            with tracer.start_as_current_span(
                f'tool:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'gen_ai.operation.name': 'execute_tool',
                    'gen_ai.tool.name': name,
                    'step.name': name,
                    'gen_ai.tool.call.arguments': input_str,
                    'gen_ai.input.messages': _msg('tool', input_str),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                span.set_attribute('gen_ai.tool.call.result', str(result))
                span.set_attribute('gen_ai.output.messages', _msg('tool', str(result)))
                return result
        return wrapper
    return decorator


def otel_llm(model_name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            messages = args[0] if args else kwargs.get('messages', [])
            with tracer.start_as_current_span(
                'chat',
                attributes={
                    'gen_ai.system': 'openai',
                    'gen_ai.operation.name': 'chat',
                    'gen_ai.request.model': model_name,
                    'gen_ai.input.messages': json.dumps(messages),
                },
            ) as span:
                response = fn(*args, **kwargs)
                content = response.choices[0].message.content
                span.set_attribute('gen_ai.output.messages', _msg('assistant', content))
                span.set_attribute('gen_ai.response.model', response.model)
                if response.usage:
                    span.set_attribute('gen_ai.usage.input_tokens', response.usage.prompt_tokens)
                    span.set_attribute('gen_ai.usage.output_tokens', response.usage.completion_tokens)
                return response
        return wrapper
    return decorator


print('Decorators defined: @otel_workflow, @otel_retriever, @otel_tool, @otel_llm')

Decorators defined: @otel_workflow, @otel_retriever, @otel_tool, @otel_llm


## Step 6: Use the Decorators in a Pipeline

Now the application code is clean Python — no tracing boilerplate. Every function is annotated with its span type, and the decorators handle the Galileo attribute contract.

The `@otel_llm` decorator wraps the raw `client.chat.completions.create(...)` call, so no OpenInference instrumentor is needed.

In [8]:
@otel_retriever('search-knowledge-base', **{'http.url': 'https://kb.internal/search', 'http.method': 'POST'})
def search_knowledge_base(query):
    return [
        {'content': 'TracerProvider owns span processors and tracers.', 'metadata': {'source': 'otel-docs'}},
        {'content': 'Galileo routes OTLP spans via Galileo-API-Key, project, and logstream headers.', 'metadata': {'source': 'galileo-docs'}},
        {'content': 'Manual LLM spans need gen_ai.operation.name=chat and gen_ai.request.model.', 'metadata': {'source': 'semconv-docs'}},
    ]


@otel_tool('format-context', **{'step.operation': 'join-lines'})
def format_context(docs):
    return '\n'.join(f'- {doc["content"]}' for doc in docs)


@otel_tool('format-final-output', **{'step.operation': 'strip'})
def format_final_output(answer):
    return answer.strip()


@otel_llm(MODEL)
def call_llm(messages):
    return client.chat.completions.create(model=MODEL, messages=messages, max_tokens=100)


@otel_workflow('knowledge-qa-pipeline', **{'usecase.id': 'qa-demo', 'sys.id': 'notebook-sys-id'})
def answer_question(query):
    docs = search_knowledge_base(query)
    context = format_context(docs)

    response = call_llm([
        {'role': 'system', 'content': 'Answer concisely based on the context provided.'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
    ])

    return format_final_output(response.choices[0].message.content)


final = answer_question('What is a TracerProvider?')
force_flush()

print(f'Answer: {final}')
print('\n-> Pipeline trace flushed to Galileo (all spans manual, no OpenInference)')
print(f'   Check: {log_stream_url}')

Answer: A TracerProvider is a component that manages span processors and tracers for collecting and reporting tracing data in distributed systems.

-> Pipeline trace flushed to Galileo (all spans manual, no OpenInference)
   Check: https://console.demo-v2.galileocloud.io/project/9ec42db0-d40c-4ee0-9f28-e6f29a83288f/log-streams/470b0435-70fd-4aca-953d-7b208f5dc7bb


## Step 7: Flush and Verify

`BatchSpanProcessor` exports asynchronously. An explicit `force_flush()` ensures all spans are visible in the Galileo console before moving on.

In [9]:
force_flush()

print('All spans flushed to Galileo')
print(f'View traces and metrics at: {log_stream_url}')

All spans flushed to Galileo
View traces and metrics at: https://console.demo-v2.galileocloud.io/project/9ec42db0-d40c-4ee0-9f28-e6f29a83288f/log-streams/470b0435-70fd-4aca-953d-7b208f5dc7bb


## Step 8: Clean Up

1. Uninstrument `requests` so later notebook cells start cleanly
2. Shut down the `TracerProvider` so the batch processor drains
3. Delete the Galileo project

In [10]:
req_instrumentor.uninstrument()
tracer_provider.shutdown()

delete_project(name=PROJECT_NAME)
print(f'Deleted project: {PROJECT_NAME}')

Deleted project: GalileoEval_S12_ManualOTel
